# PipTransformer: a from-scratch decoder-only Transformer on TinyStories

A compact, fully readable implementation of a Llama-style language model, trained from scratch on the
[TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) corpus.

The notebook covers the complete pipeline:

1. Train a Byte-Level BPE tokenizer.
2. Build the Transformer (RMSNorm, RoPE, causal multi-head attention, SwiGLU).
3. Run a sanity check on the initial loss.
4. Tokenize the corpus into memory-mapped token streams.
5. Train with gradient accumulation, mixed precision and a warmup + cosine schedule.
6. Save / resume checkpoints.
7. Visualize attention and generate stories.

## Reference results

These numbers come from the original 2,000-step run on an Apple Silicon GPU. Re-running the notebook
reproduces the pipeline; see `README.md` for the full table and a sample generation.

| Metric | Value |
| --- | --- |
| Parameters | 31,465,984 |
| Layers / heads / width | 8 / 8 / 512 |
| Context length | 512 |
| Final validation loss | 1.7293 |
| Final validation perplexity | 5.64 |
| Training throughput | ~23k-28k tokens/s (MPS) |

## How to run

Run the cells top to bottom. On first use the notebook downloads TinyStories and trains the tokenizer;
both are cached (under `data/` and `tokenizer/`) so later runs start immediately.

> **Hardware:** the notebook automatically selects Apple Silicon (MPS), NVIDIA CUDA, or CPU.


## Contents

1. [Environment setup](#1-environment-setup)
2. [Configuration](#2-configuration)
3. [Dataset and tokenizer](#3-dataset-and-byte-level-bpe-tokenizer)
4. [Model architecture](#4-model-architecture)
5. [Instantiate the model](#5-instantiate-the-model)
6. [Sampling utility](#6-sampling-utility)
7. [Sanity check](#7-sanity-check-a-random-model-should-start-near-uniform-loss)
8. [Before training](#8-before-training-the-model-babbles)
9. [Token cache](#9-tokenize-the-corpus-into-flat-memmapped-token-streams)
10. [Training loop](#10-training-loop)
11. [Save a checkpoint](#11-save-a-checkpoint)
12. [Resume a checkpoint](#12-resume-from-a-checkpoint)
13. [Attention inspection](#13-inspect-attention-patterns)
14. [Generation](#14-generate-after-training-or-after-loading-a-checkpoint)
15. [Optional graph](#15-optional-render-the-computation-graph)


## 1. Environment setup

The notebook selects the best available device automatically and seeds every random-number generator
for reproducible runs.


In [ ]:
# ============================================================================
# 1. Environment setup
# ============================================================================
# The notebook is deliberately written with plain PyTorch so that every part
# of the model can be inspected. Run it top to bottom the first time.

# Python standard library ----------------------------------------------------
import os            # filesystem paths and environment variables
import math          # log/exp/sqrt helpers (loss, perplexity, RoPE)
import random        # Python RNG seeding
import time          # wall-clock timing for the training loop
import hashlib       # SHA-256 fingerprints for tokenizer/cache validation
import json          # reading/writing the token-cache metadata
from dataclasses import dataclass, asdict  # typed config + checkpoint metadata

# Third-party ----------------------------------------------------------------
import numpy as np   # memmapped token streams and batch sampling
import torch         # tensors, autograd, optimizers
import torch.nn as nn               # Module, Linear, Embedding
import torch.nn.functional as F     # silu, softmax, scaled_dot_product_attention
from datasets import load_dataset   # TinyStories download / caching
from tokenizers import (            # HuggingFace tokenizers (Rust backend)
    Tokenizer,
    trainers,
    models,
    pre_tokenizers,
    decoders,
)

# Device selection -----------------------------------------------------------
# MPS (Apple Silicon) is checked first so local Macs use the GPU by default.
# CUDA is used on NVIDIA machines, and CPU is the always-available fallback.
if torch.backends.mps.is_available():
    device = torch.device("mps")
    DEVICE_NAME = "Apple Silicon GPU (MPS)"
elif torch.cuda.is_available():
    device = torch.device("cuda")
    DEVICE_NAME = f"NVIDIA CUDA GPU ({torch.cuda.get_device_name(0)})"
else:
    device = torch.device("cpu")
    DEVICE_NAME = "CPU"
print(f"Using {DEVICE_NAME}")


def set_seed(seed: int = 42) -> int:
    """Seed every RNG used by the notebook and return the seed.

    Notes
    -----
    * ``PYTHONHASHSEED`` must be set before the interpreter starts to fully
      affect hashing; it is still set here for subprocesses.
    * Full bitwise determinism is not guaranteed on GPU/MPS, but this removes
      the most common sources of run-to-run variation.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)  # Python hash randomization
    random.seed(seed)                          # stdlib random
    np.random.seed(seed)                       # NumPy legacy RNG (used by datasets)
    torch.manual_seed(seed)                    # CPU torch RNG
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    return seed


SEED = 42
set_seed(SEED)
print(f"Random seed fixed to {SEED}")


## 2. Configuration

Every path and hyperparameter lives in one typed dataclass. The same object is serialized into each
checkpoint, so a run can be resumed without guessing its architecture or optimizer settings.


In [ ]:
# ============================================================================
# 2. Configuration
# ============================================================================
@dataclass
class Config:
    """Single source of truth for paths and hyperparameters."""

    # --- Data / tokenizer -------------------------------------------------
    dataset_name: str = "roneneldan/TinyStories"
    tokenizer_path: str = "tokenizer/tokenizer.json"
    # The tokenizer only needs a representative sample, so training it on 50k
    # stories is a deliberate speed/quality tradeoff (training uses 80k).
    tokenizer_train_slice: str = "train[:50000]"
    train_slice: str = "train[:80000]"
    val_slice: str = "validation[:2000]"
    vocab_size: int = 8192

    # --- Model ------------------------------------------------------------
    dim: int = 512          # residual stream width
    n_layers: int = 8       # number of Transformer blocks
    n_heads: int = 8        # attention heads per block
    hidden_dim: int = 1536  # SwiGLU hidden width (3x dim, as in Llama)
    max_seq_len: int = 512  # context length and RoPE cache size

    # --- Optimization -----------------------------------------------------
    batch_size: int = 16          # sequences per micro-batch
    grad_accum_steps: int = 4     # micro-batches per optimizer step
    max_steps: int = 2000         # total optimizer steps
    eval_interval: int = 200      # validation cadence
    log_interval: int = 50        # training-log cadence
    learning_rate: float = 1e-3   # peak learning rate
    min_lr: float = 1e-4          # final learning rate after cosine decay
    warmup_steps: int = 200       # linear warmup length
    weight_decay: float = 0.1     # AdamW decoupled weight decay
    beta1: float = 0.9            # Adam first-moment decay
    beta2: float = 0.95           # Adam second-moment decay
    grad_clip: float = 1.0        # global gradient-norm limit
    val_batches: int = 8          # fixed validation batches per evaluation

    # --- Output -----------------------------------------------------------
    data_dir: str = "data"
    checkpoint_dir: str = "checkpoints"
    checkpoint_name: str = "pip_transformer_final.pt"

    # --- Generation -------------------------------------------------------
    temperature: float = 0.8
    top_k: int = 50
    max_new_tokens: int = 120

    def __post_init__(self) -> None:
        # Attention splits ``dim`` evenly across heads.
        if self.dim % self.n_heads != 0:
            raise ValueError("dim must be divisible by n_heads")
        # RoPE rotates pairs of channels, so each head dimension must be even.
        if (self.dim // self.n_heads) % 2 != 0:
            raise ValueError("head_dim must be even for RoPE")
        # Token streams are stored as uint16, so vocabulary ids must fit.
        if self.vocab_size >= 65536:
            raise ValueError("vocab_size must be < 65536 for uint16 storage")

    @property
    def head_dim(self) -> int:
        """Per-head channel width."""
        return self.dim // self.n_heads

    @property
    def effective_batch_size(self) -> int:
        """Number of sequences contributing to one optimizer step."""
        return self.batch_size * self.grad_accum_steps

    @property
    def tokens_per_step(self) -> int:
        """Number of training tokens consumed per optimizer step."""
        return self.effective_batch_size * self.max_seq_len

    @property
    def checkpoint_path(self) -> str:
        """Full path to the default checkpoint file."""
        return os.path.join(self.checkpoint_dir, self.checkpoint_name)


# Instantiate the single config object used everywhere below.
cfg = Config()
print(cfg)


## 3. Dataset and Byte-Level BPE tokenizer

TinyStories is a synthetic corpus of short, simple English stories. It is small enough to train on a
laptop but rich enough to demonstrate real language modelling.

The tokenizer is a **Byte-Level BPE** model. Byte-level pre-tokenization guarantees that any UTF-8
text can be represented (no unknown characters), while BPE learns frequent sub-word merges. The
matching `ByteLevel` decoder is essential: without it, `decode()` leaves the internal space marker
`Ġ` and newline marker `Ċ` in the output.


In [ ]:
# ============================================================================
# 3. Dataset and Byte-Level BPE tokenizer
# ============================================================================
# TinyStories is a synthetic corpus of short, simple English stories. It is
# small enough to train a useful model on a laptop, yet rich enough to
# demonstrate real language modelling.
print("Loading TinyStories for tokenizer training...")
tokenizer_corpus = load_dataset(cfg.dataset_name, split=cfg.tokenizer_train_slice)


def train_tokenizer(path: str, corpus, vocab_size: int) -> Tokenizer:
    """Train a Byte-Level BPE tokenizer and save it to ``path``.

    Byte-Level BPE never produces unknown tokens for arbitrary UTF-8 text
    because every byte is representable. ``ByteLevel`` marks a leading space
    with the special character ``Ġ``; the matching decoder turns it back into a
    real space.
    """
    def batch_iterator(batch_size: int = 10_000):
        # ``datasets`` rows are dictionaries; the trainer only needs the raw
        # strings, so yield chunks instead of materializing the whole corpus.
        for start in range(0, len(corpus), batch_size):
            yield corpus[start:start + batch_size]["text"]

    tok = Tokenizer(models.BPE(unk_token="<unk>"))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

    # IMPORTANT: without a decoder, ``decode()`` leaves ``Ġ``/``Ċ`` byte-level
    # markers in the output. Setting it here makes decoding lossless and
    # human-readable (spaces and newlines are restored).
    tok.decoder = decoders.ByteLevel()

    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        # Special tokens get the lowest ids; <|endoftext|> is id 0.
        special_tokens=["<|endoftext|>", "<user>", "<assistant>", "<unk>"],
        show_progress=True,
    )
    tok.train_from_iterator(batch_iterator(), trainer=trainer)

    # Persist the tokenizer so later runs (and checkpoints) use exactly the
    # same vocabulary.
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tok.save(path)
    print(f"Tokenizer trained and saved to {path}")
    return tok


def load_or_train_tokenizer(cfg: Config) -> Tokenizer:
    """Load the tokenizer from disk, training it on first use."""
    if os.path.exists(cfg.tokenizer_path):
        print(f"Loading tokenizer from {cfg.tokenizer_path}")
        tok = Tokenizer.from_file(cfg.tokenizer_path)
    else:
        print(f"No tokenizer found at {cfg.tokenizer_path}; training one...")
        tok = train_tokenizer(cfg.tokenizer_path, tokenizer_corpus, cfg.vocab_size)

    # Older tokenizer files (including the one used to produce the original
    # results) were saved with ``"decoder": null``. Re-attach the decoder in
    # memory so decoding is correct even for those files.
    if tok.decoder is None:
        tok.decoder = decoders.ByteLevel()
    return tok


tokenizer = load_or_train_tokenizer(cfg)
EOS_ID = tokenizer.token_to_id("<|endoftext|>")
if EOS_ID is None:
    raise ValueError("<|endoftext|> is missing from the tokenizer's special tokens")
print(f"Vocabulary size: {tokenizer.get_vocab_size():,} | EOS token id: {EOS_ID}")

# Smoke test: encoding then decoding should round-trip spaces and newlines.
_probe = "Hello world!\nOnce upon a time, there was a tiny robot."
_probe_ids = tokenizer.encode(_probe).ids
_probe_text = tokenizer.decode(_probe_ids, skip_special_tokens=True)
print("Encode/decode round-trip:", repr(_probe_text))
assert _probe_text == _probe, "tokenizer round-trip failed; check the decoder"


## 4. Model architecture

The model follows the Llama-style decoder-only Transformer recipe:

- **Pre-norm** residual blocks (normalization inside each residual branch).
- **RoPE** (rotary position embeddings) instead of learned position vectors.
- **SwiGLU** feed-forward network instead of a plain two-layer MLP.
- **No biases** in any linear layer.
- **Tied** input and output embeddings.

Tensor shapes are annotated as `(B, T, C)` = `(batch, time/sequence, channels/dim)`.

### RMSNorm

Normalizes each token vector by its root-mean-square magnitude:

```
RMSNorm(x) = x / sqrt(mean(x^2) + eps) * weight
```

It skips mean subtraction and the bias used by LayerNorm, which is cheaper and works well in modern
decoders.

### Rotary position embeddings (RoPE)

For each position `m` and frequency `f_i`, RoPE rotates consecutive channel pairs by the phase
`m * f_i`. After rotation, the dot product of two queries/keys depends only on their **relative**
offset, which is exactly what attention needs. Phases are precomputed once and reused; they are not
learned.

### Causal attention

Queries and keys are rotated, then `F.scaled_dot_product_attention(..., is_causal=True)` applies the
lower-triangular mask inside the fused kernel. No explicit `(T, T)` mask buffer is stored.


In [ ]:
# ============================================================================
# 4. Model architecture
# ============================================================================
# Shapes are annotated as (B, T, C) = (batch, sequence length, channels).


class RMSNorm(nn.Module):
    """Root-Mean-Square normalization.

    Computes ``x / sqrt(mean(x^2) + eps) * weight`` along the last dimension.
    Unlike LayerNorm it does not subtract the mean or learn a bias; this is
    cheaper and works well in modern decoder-only Transformers.

    Parameters
    ----------
    dim:
        Size of the last (channel) dimension.
    eps:
        Small constant for numerical stability.
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # Learnable per-channel scale, initialized to identity (all ones).
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (..., dim). Average the squared values over the channel axis only.
        variance = x.pow(2).mean(dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(variance + self.eps)
        return x_normed * self.weight


def precompute_rope_frequencies(
    dim: int, max_seq_len: int = 512, theta: float = 10000.0
) -> torch.Tensor:
    """Precompute the complex exponentials used by RoPE.

    For each position ``m`` and frequency ``f_i`` we store ``exp(i * m * f_i)``.
    Rotating a real query/key pair by this phase encodes absolute position,
    while the dot product between two rotated vectors depends only on their
    relative offset ``m - n``.

    Returns
    -------
    torch.Tensor
        Complex tensor of shape ``(max_seq_len, dim // 2)``.
    """
    # Standard RoPE frequencies: theta^(-2i/dim), i = 0 .. dim/2 - 1.
    inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    positions = torch.arange(max_seq_len, dtype=torch.float32)
    angles = torch.outer(positions, inv_freq)          # (max_seq_len, dim/2)
    # e^{i*angle} = cos(angle) + i*sin(angle), with unit magnitude.
    return torch.polar(torch.ones_like(angles), angles)


def apply_rope(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """Rotate ``x`` in consecutive 2-D pairs according to position.

    Parameters
    ----------
    x:
        Q or K tensor of shape ``(B, T, H, D)`` where ``D`` is the head dim.
    freqs_cis:
        Output of :func:`precompute_rope_frequencies`, shape
        ``(max_seq_len, D // 2)``.
    """
    T = x.shape[1]
    if T > freqs_cis.shape[0]:
        raise ValueError(
            f"sequence length {T} exceeds RoPE cache {freqs_cis.shape[0]}"
        )

    # View the last dimension as complex numbers: (B, T, H, D/2).
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    # Broadcast the cached phases across batch and heads: (1, T, 1, D/2).
    freqs = freqs_cis[:T].unsqueeze(0).unsqueeze(2)
    rotated = x_complex * freqs
    # Back to real values with the original shape and dtype.
    return torch.view_as_real(rotated).flatten(3).type_as(x)


class Attention(nn.Module):
    """Causal multi-head self-attention with RoPE.

    The four projections are bias-free, matching modern Llama-style models.
    ``F.scaled_dot_product_attention`` fuses the softmax, causal mask and
    value aggregation, so no explicit ``(T, T)`` mask is needed.
    """
    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        if dim % n_heads != 0:
            raise ValueError("dim must be divisible by n_heads")
        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        # Query, key, value and output projections (all bias-free).
        self.wq = nn.Linear(dim, dim, bias=False)
        self.wk = nn.Linear(dim, dim, bias=False)
        self.wv = nn.Linear(dim, dim, bias=False)
        self.wo = nn.Linear(dim, dim, bias=False)

    def forward(self, x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape  # batch, sequence length, channels

        # Project and split the channel dimension into heads.
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim)
        k = self.wk(x).view(B, T, self.n_heads, self.head_dim)
        v = self.wv(x).view(B, T, self.n_heads, self.head_dim)

        # Rotate Q and K only; V carries content and must not be rotated.
        q = apply_rope(q, freqs_cis)
        k = apply_rope(k, freqs_cis)

        # SDPA expects (B, H, T, D).
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # ``is_causal=True`` applies the lower-triangular mask internally.
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        # Merge heads back into channels: (B, T, C).
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.wo(out)


class SwiGLUMLP(nn.Module):
    """Gated feed-forward network used in Llama-style Transformers.

    ``SwiGLU(x) = W_down( SiLU(W_gate x) * (W_up x) )``. The gate lets the
    network modulate information flow multiplicatively, which empirically
    improves quality over a plain two-layer MLP at the same parameter budget.
    """
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # SiLU (a.k.a. swish) on the gate, multiplied element-wise by the up
        # projection, then projected back to the model width.
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class TransformerBlock(nn.Module):
    """Pre-norm decoder block: attention then SwiGLU, with residual adds."""
    def __init__(self, dim: int, n_heads: int, hidden_dim: int):
        super().__init__()
        self.attention = Attention(dim, n_heads)
        self.mlp = SwiGLUMLP(dim, hidden_dim)
        # Normalization happens inside the residual branch (pre-norm).
        self.attention_norm = RMSNorm(dim)
        self.ffn_norm = RMSNorm(dim)

    def forward(self, x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
        # Residual 1: token mixing (attention).
        x = x + self.attention(self.attention_norm(x), freqs_cis)
        # Residual 2: channel mixing (feed-forward).
        x = x + self.mlp(self.ffn_norm(x))
        return x


class PipTransformer(nn.Module):
    """A small Llama-style decoder-only Transformer.

    Parameters
    ----------
    vocab_size:
        Number of BPE tokens (8192 for this project).
    dim, n_layers, n_heads, hidden_dim:
        Width, depth, head count and SwiGLU hidden width.
    max_seq_len:
        Maximum context length; also the size of the RoPE cache.
    """
    def __init__(
        self,
        vocab_size: int = 8192,
        dim: int = 512,
        n_layers: int = 8,
        n_heads: int = 8,
        hidden_dim: int = 1536,
        max_seq_len: int = 512,
    ):
        super().__init__()
        self.max_seq_len = max_seq_len

        self.tok_embeddings = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList(
            [TransformerBlock(dim, n_heads, hidden_dim) for _ in range(n_layers)]
        )
        self.norm = RMSNorm(dim)
        self.output = nn.Linear(dim, vocab_size, bias=False)

        # Weight tying: the output projection reuses the embedding matrix.
        # This saves ``vocab_size * dim`` parameters and often improves quality.
        self.output.weight = self.tok_embeddings.weight

        # RoPE phases are fixed (not learned). ``persistent=False`` keeps them
        # out of the checkpoint because they are deterministic.
        freqs_cis = precompute_rope_frequencies(dim // n_heads, max_seq_len)
        self.register_buffer("freqs_cis", freqs_cis, persistent=False)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """Map token ids ``(B, T)`` to logits ``(B, T, vocab_size)``."""
        if idx.shape[1] > self.max_seq_len:
            raise ValueError(
                f"input length {idx.shape[1]} exceeds max_seq_len {self.max_seq_len}"
            )

        h = self.tok_embeddings(idx)      # (B, T, C)
        for layer in self.layers:         # repeated block application
            h = layer(h, self.freqs_cis)
        h = self.norm(h)                  # final RMSNorm
        return self.output(h)             # logits over the vocabulary


## 5. Instantiate the model

The default configuration (`dim=512`, 8 layers, 8 heads, `hidden_dim=1536`) yields **31,465,984**
parameters. Because the output projection is tied to the token embedding, initializing the embedding
also initializes the output head.


In [ ]:
# ============================================================================
# 5. Instantiate the model
# ============================================================================
set_seed(SEED)
model = PipTransformer(
    vocab_size=cfg.vocab_size,
    dim=cfg.dim,
    n_layers=cfg.n_layers,
    n_heads=cfg.n_heads,
    hidden_dim=cfg.hidden_dim,
    max_seq_len=cfg.max_seq_len,
).to(device)

# Small-std normal initialization for the embedding matrix (GPT-2 style).
# Note: ``output.weight`` is tied to ``tok_embeddings.weight``, so this also
# initializes the output projection.
nn.init.normal_(model.tok_embeddings.weight, mean=0.0, std=0.02)

n_params = sum(p.numel() for p in model.parameters())
print(f"PipTransformer initialized with {n_params:,} parameters on {device}")

# Sanity check: the model should accept a short batch and return vocab logits.
with torch.no_grad():
    _dummy_logits = model(torch.zeros((2, 16), dtype=torch.long, device=device))
print(f"Forward pass OK: input (2, 16) -> logits {tuple(_dummy_logits.shape)}")


## 6. Sampling utility

A single reusable sampler is used both before training (to show the random model) and after training
(to generate stories). It supports temperature scaling, top-k filtering and an early stop on the
end-of-text token.


In [ ]:
# ============================================================================
# 6. Sampling utility
# ============================================================================
@torch.no_grad()
def generate(
    model: nn.Module,
    tokenizer: Tokenizer,
    prompt: str,
    *,
    max_new_tokens: int = 80,
    temperature: float = 0.8,
    top_k: int | None = 50,
    seed: int = 42,
) -> str:
    """Autoregressively sample a completion for ``prompt``.

    Parameters
    ----------
    temperature:
        Logits are divided by this value before softmax. Lower = more
        deterministic, higher = more random. Must be positive.
    top_k:
        Keep only the ``k`` most likely tokens at each step. ``None`` disables
        the filter. This prevents sampling vanishingly unlikely tokens.
    seed:
        Seed for the sampling generator so results are reproducible.
    """
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    model.eval()

    eos_id = tokenizer.token_to_id("<|endoftext|>")
    ids = tokenizer.encode(prompt).ids
    x = torch.tensor([ids], dtype=torch.long, device=device)

    # ``torch.multinomial`` draws from the categorical distribution defined by
    # the softmax probabilities. A dedicated generator keeps this reproducible.
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)

    for _ in range(max_new_tokens):
        # Clip to the model's context window before the forward pass.
        context = x[:, -model.max_seq_len:]
        logits = model(context)[:, -1, :]  # logits for the next token

        if temperature != 1.0:
            logits = logits / temperature

        if top_k is not None and top_k > 0:
            k = min(top_k, logits.size(-1))
            kth_value = torch.topk(logits, k, dim=-1).values[:, [-1]]
            logits = logits.masked_fill(logits < kth_value, -float("inf"))

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1, generator=gen)

        # ``<|endoftext|>`` marks story boundaries in the training stream.
        if eos_id is not None and next_id.item() == eos_id:
            break
        x = torch.cat([x, next_id], dim=1)

    # ``skip_special_tokens`` removes any remaining control tokens; the
    # ByteLevel decoder restores spaces and newlines.
    return tokenizer.decode(x[0].tolist(), skip_special_tokens=True)


## 7. Sanity check: a random model should start near uniform loss

At initialization the model knows nothing, so its predictions are approximately uniform over the
vocabulary. The cross-entropy of a uniform distribution over `V` tokens is `ln(V)`; for `V = 8192`
this is about **9.01**. A loss near that value confirms the wiring is correct.


In [ ]:
# ============================================================================
# 7. Sanity check: a random model should start near uniform loss
# ============================================================================
# At initialization the model knows nothing, so the cross-entropy of a uniform
# distribution over V tokens is ln(V). For V = 8192 this is ~9.01.
def get_sanity_tokens(text: str, tokenizer: Tokenizer, seq_len: int) -> list[int]:
    """Tokenize ``text`` and pad by repetition until it has ``seq_len + 1`` ids."""
    ids = tokenizer.encode(text).ids
    if not ids:
        raise ValueError("the sample text produced no tokens")
    if len(ids) < seq_len + 1:
        ids = ids * (seq_len // len(ids) + 2)  # +2 guarantees enough copies
    return ids


criterion = nn.CrossEntropyLoss()  # mean over all token positions

sample_text = tokenizer_corpus[0]["text"]
sanity_ids = get_sanity_tokens(sample_text, tokenizer, cfg.max_seq_len)
x = torch.tensor([sanity_ids[: cfg.max_seq_len]], dtype=torch.long, device=device)
y = torch.tensor([sanity_ids[1 : cfg.max_seq_len + 1]], dtype=torch.long, device=device)

# Use a temporary optimizer so this smoke test does not disturb the main
# training state (which is created later).
sanity_optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate)

model.train()
sanity_optimizer.zero_grad(set_to_none=True)
logits = model(x)  # (1, T, vocab)
loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip)
sanity_optimizer.step()

expected = math.log(cfg.vocab_size)
print(
    f"Initial sanity-check loss: {loss.item():.4f} "
    f"(expected ~{expected:.2f} for V={cfg.vocab_size})"
)

del sanity_optimizer


## 8. Before training: the model babbles

With random weights, sampling produces nonsense. Running this cell makes the improvement after
training easy to see.


In [ ]:
# ============================================================================
# 8. Before training: the model babbles
# ============================================================================
prompt = "Once upon a time, a little girl named Lily"
print("Prompt:", prompt)
print("\n--- Untrained model ---")
print(
    generate(
        model,
        tokenizer,
        prompt,
        max_new_tokens=60,
        temperature=0.9,
        top_k=50,
        seed=SEED,
    )
)


## 9. Tokenize the corpus into flat, memmapped token streams

Tokenizing text on the fly would make training I/O-bound. Instead, each story is tokenized once,
followed by an end-of-text separator, and the whole split is written to a flat binary file of `uint16`
ids. Training then memory-maps the file and samples random windows.

The cache is validated against the tokenizer hash and the dataset slices, so changing either triggers
a rebuild automatically.


In [ ]:
# ============================================================================
# 9. Tokenize the corpus into flat, memmapped token streams
# ============================================================================
# Training on a single contiguous stream of uint16 token ids is much faster
# than tokenizing text on the fly. We:
#   1. tokenize each story and append <|endoftext|> as a separator,
#   2. concatenate everything into one binary file per split,
#   3. memory-map the files and sample random windows for training.
os.makedirs(cfg.data_dir, exist_ok=True)
train_bin = os.path.join(cfg.data_dir, "train.bin")
val_bin = os.path.join(cfg.data_dir, "val.bin")
meta_path = os.path.join(cfg.data_dir, "meta.json")


def tokenizer_fingerprint(path: str) -> str:
    """SHA-256 of the tokenizer file, used to invalidate stale token caches."""
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def build_token_cache() -> dict:
    """Download, tokenize and write the train/val token streams."""
    print("Preparing training and validation datasets...")
    train_data = load_dataset(cfg.dataset_name, split=cfg.train_slice)
    val_data = load_dataset(cfg.dataset_name, split=cfg.val_slice)

    def encode_stories(batch):
        # ``batch["text"]`` is a list of stories. Tokenize all of them and
        # insert EOS after each story so the model learns document boundaries.
        all_ids: list[int] = []
        for encoded in tokenizer.encode_batch(batch["text"]):
            all_ids.extend(encoded.ids)
            all_ids.append(EOS_ID)
        return {"input_ids": all_ids}

    print("Tokenizing datasets...")
    train_encoded = train_data.map(
        encode_stories, batched=True, remove_columns=["text"]
    )
    val_encoded = val_data.map(
        encode_stories, batched=True, remove_columns=["text"]
    )

    def write_ids(path: str, rows) -> int:
        """Write token ids to disk as uint16 and return the total count."""
        total = 0
        with open(path, "wb") as f:
            if len(rows) and isinstance(rows[0], (list, tuple, np.ndarray)):
                # A batched map can return a list of lists (one per batch).
                for start in range(0, len(rows), 10_000):
                    chunk = rows[start : start + 10_000]
                    ids = np.concatenate(
                        [np.asarray(row, dtype=np.uint16) for row in chunk]
                    )
                    if ids.size and int(ids.max()) >= cfg.vocab_size:
                        raise ValueError("token id does not fit the vocabulary")
                    ids.tofile(f)
                    total += int(ids.size)
            else:
                # Flat list of ints (one token stream for the whole split).
                for start in range(0, len(rows), 1_000_000):
                    ids = np.asarray(
                        rows[start : start + 1_000_000], dtype=np.uint16
                    )
                    if ids.size and int(ids.max()) >= cfg.vocab_size:
                        raise ValueError("token id does not fit the vocabulary")
                    ids.tofile(f)
                    total += int(ids.size)
        return total

    n_train = write_ids(train_bin, train_encoded["input_ids"])
    n_val = write_ids(val_bin, val_encoded["input_ids"])

    meta = {
        "format_version": 1,
        "train_tokens": n_train,
        "val_tokens": n_val,
        "dtype": "uint16",
        "vocab_size": cfg.vocab_size,
        "train_slice": cfg.train_slice,
        "val_slice": cfg.val_slice,
        "tokenizer_fingerprint": tokenizer_fingerprint(cfg.tokenizer_path),
    }
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    return meta


# Reuse the cache only when every relevant setting still matches.
meta = None
if os.path.exists(meta_path):
    with open(meta_path, encoding="utf-8") as f:
        meta = json.load(f)

cache_ok = (
    meta is not None
    and os.path.exists(train_bin)
    and os.path.exists(val_bin)
    and meta.get("format_version") == 1
    and meta.get("vocab_size") == cfg.vocab_size
    and meta.get("train_slice") == cfg.train_slice
    and meta.get("val_slice") == cfg.val_slice
    and meta.get("tokenizer_fingerprint")
    == tokenizer_fingerprint(cfg.tokenizer_path)
)
if cache_ok:
    print(
        f"Using cached tokens: {meta['train_tokens']:,} train / "
        f"{meta['val_tokens']:,} val"
    )
else:
    meta = build_token_cache()

# ``np.memmap`` keeps the token files on disk and pages data in on demand.
train_tokens = np.memmap(train_bin, dtype=np.uint16, mode="r")
val_tokens = np.memmap(val_bin, dtype=np.uint16, mode="r")
print(f"Token streams: train={len(train_tokens):,} | val={len(val_tokens):,}")


In [ ]:
# ============================================================================
# Batch sampling
# ============================================================================
def get_batch(
    data_stream, batch_size: int, seq_len: int, rng: np.random.Generator
):
    """Sample ``batch_size`` random windows and return (inputs, targets).

    Targets are inputs shifted one token to the right, which is the standard
    next-token-prediction objective. Both tensors have shape ``(B, T)``.
    """
    max_idx = len(data_stream) - seq_len - 1
    if max_idx <= 0:
        raise ValueError("token stream is shorter than seq_len")
    starts = rng.integers(0, max_idx, size=batch_size)
    x = np.stack([data_stream[i : i + seq_len] for i in starts]).astype(np.int64)
    y = np.stack(
        [data_stream[i + 1 : i + seq_len + 1] for i in starts]
    ).astype(np.int64)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)


set_seed(SEED)
train_rng = np.random.default_rng(SEED)      # training-window sampler
val_rng = np.random.default_rng(SEED + 1)    # validation-window sampler

# Pre-sample a fixed validation set. Evaluating the same batches every time
# makes validation-loss comparisons across steps meaningful.
val_batches = [
    get_batch(val_tokens, cfg.batch_size, cfg.max_seq_len, val_rng)
    for _ in range(cfg.val_batches)
]
print(
    f"Cached {len(val_batches)} validation batches of shape "
    f"({cfg.batch_size}, {cfg.max_seq_len})"
)


## 10. Training loop

Training uses:

- **AdamW** with decoupled weight decay.
- **Gradient accumulation** (`16 x 4 = 64` sequences per optimizer step) to simulate a larger batch.
- **Linear warmup** for the first 200 steps, then **cosine decay** to `1e-4`.
- **Gradient clipping** at norm 1.0 for stability.
- **Mixed precision** on accelerators (`bf16` on MPS, `fp16` + loss scaling on CUDA).

Validation uses a fixed set of batches so the loss curve is directly comparable across checkpoints.


In [ ]:
# ============================================================================
# 10. Training loop
# ============================================================================
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.learning_rate,
    betas=(cfg.beta1, cfg.beta2),
    weight_decay=cfg.weight_decay,
)


def get_lr(step: int) -> float:
    """Learning-rate schedule: linear warmup, then cosine decay to ``min_lr``."""
    if step < cfg.warmup_steps:
        # Warmup avoids large, unstable updates in the first few steps.
        return cfg.learning_rate * (step / cfg.warmup_steps)
    if step >= cfg.max_steps or cfg.max_steps <= cfg.warmup_steps:
        return cfg.min_lr
    # Fraction of the decay phase elapsed, in [0, 1].
    decay_ratio = (step - cfg.warmup_steps) / (
        cfg.max_steps - cfg.warmup_steps
    )
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))  # 1 -> 0
    return cfg.min_lr + coeff * (cfg.learning_rate - cfg.min_lr)


# Mixed-precision setup ------------------------------------------------------
if device.type == "mps":
    amp_dtype = torch.bfloat16  # MPS bf16 autocast (recent PyTorch)
    scaler = None               # bf16 does not need loss scaling
elif device.type == "cuda":
    amp_dtype = torch.float16   # fp16 needs loss scaling
    # ``torch.amp.GradScaler`` is newer; keep a fallback for older releases.
    if hasattr(torch.amp, "GradScaler"):
        scaler = torch.amp.GradScaler("cuda")
    else:
        scaler = torch.cuda.amp.GradScaler()
else:
    amp_dtype = None            # CPU stays in fp32
    scaler = None


def forward_loss(x, y):
    """Run a forward pass and return the mean cross-entropy loss."""
    logits = model(x)
    return criterion(logits.view(-1, logits.size(-1)), y.view(-1))


model.train()
print(f"Starting training on {DEVICE_NAME} for {cfg.max_steps} steps")
print(
    f"Effective batch size: {cfg.effective_batch_size} sequences "
    f"({cfg.tokens_per_step:,} tokens/step)"
)
start_time = time.time()

for step in range(1, cfg.max_steps + 1):
    # --- learning-rate schedule -----------------------------------------
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    # --- accumulate gradients over micro-batches ------------------------
    optimizer.zero_grad(set_to_none=True)
    step_loss = torch.zeros((), device=device)

    for _ in range(cfg.grad_accum_steps):
        x, y = get_batch(
            train_tokens, cfg.batch_size, cfg.max_seq_len, train_rng
        )

        if amp_dtype is not None:
            with torch.autocast(device_type=device.type, dtype=amp_dtype):
                loss = forward_loss(x, y)
        else:
            loss = forward_loss(x, y)

        # Scale by 1/accum so the accumulated gradient equals the mean over
        # the full effective batch.
        scaled_loss = loss / cfg.grad_accum_steps
        if scaler is not None:
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()
        step_loss += loss.detach()

    # --- optimizer step --------------------------------------------------
    if scaler is not None:
        scaler.unscale_(optimizer)  # unscale before clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip)
    if scaler is not None:
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()

    # --- periodic logging ------------------------------------------------
    if step % cfg.log_interval == 0:
        elapsed = time.time() - start_time
        avg_loss = (step_loss / cfg.grad_accum_steps).item()
        perplexity = math.exp(min(avg_loss, 20))  # cap to avoid overflow
        tokens_per_sec = step * cfg.tokens_per_step / elapsed
        print(
            f"Step {step}/{cfg.max_steps} | Loss: {avg_loss:.4f} | "
            f"Perplexity: {perplexity:.2f} | LR: {lr:.5f} | "
            f"Time: {elapsed:.1f}s | Tokens/s: {tokens_per_sec:,.0f}"
        )

    # --- periodic validation --------------------------------------------
    if step % cfg.eval_interval == 0:
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for val_x, val_y in val_batches:
                if amp_dtype is not None:
                    with torch.autocast(
                        device_type=device.type, dtype=amp_dtype
                    ):
                        val_loss_batch = forward_loss(val_x, val_y)
                else:
                    val_loss_batch = forward_loss(val_x, val_y)
                val_loss += val_loss_batch.item()
        val_loss /= len(val_batches)
        print(
            f"  Validation | Loss: {val_loss:.4f} | "
            f"Perplexity: {math.exp(min(val_loss, 20)):.2f}"
        )
        model.train()

print("Training complete.")


## 11. Save a checkpoint

The checkpoint stores the model and optimizer state, the full configuration, the tokenizer
fingerprint and the PyTorch version. That is enough to rebuild the exact same model later.

> Checkpoints are large (~360 MB for this configuration) and are excluded from version control by
> the repo's `.gitignore`.


In [ ]:
# ============================================================================
# 11. Save a checkpoint
# ============================================================================
os.makedirs(cfg.checkpoint_dir, exist_ok=True)

checkpoint = {
    "format_version": 1,
    "step": cfg.max_steps,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": asdict(cfg),  # dataclass -> plain dict for portable loading
    "tokenizer_fingerprint": tokenizer_fingerprint(cfg.tokenizer_path),
    "torch_version": torch.__version__,
}
torch.save(checkpoint, cfg.checkpoint_path)
print(
    f"Saved checkpoint to {cfg.checkpoint_path} "
    f"({os.path.getsize(cfg.checkpoint_path) / 1e6:.1f} MB)"
)


## 12. Resume from a checkpoint

This cell rebuilds the model and optimizer from the checkpoint's stored config. It is safe to run right
after training (it restores the same state) and is the correct entry point after restarting the
kernel.


In [ ]:
# ============================================================================
# 12. Resume from a checkpoint
# ============================================================================
checkpoint_path = cfg.checkpoint_path  # or point this at another .pt file

# ``weights_only=False`` is required because the checkpoint contains a plain
# Python dict (the config). Only load checkpoints you trust.
checkpoint = torch.load(
    checkpoint_path, map_location=device, weights_only=False
)

hp = checkpoint["config"]
model = PipTransformer(
    vocab_size=hp["vocab_size"],
    dim=hp["dim"],
    n_layers=hp["n_layers"],
    n_heads=hp["n_heads"],
    hidden_dim=hp["hidden_dim"],
    max_seq_len=hp["max_seq_len"],
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=hp["learning_rate"],
    betas=(hp["beta1"], hp["beta2"]),
    weight_decay=hp["weight_decay"],
)
# The optimizer state is optional: if it is missing, the weights still load
# and training can continue with fresh Adam moments.
if "optimizer_state_dict" in checkpoint:
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_step = checkpoint["step"]
model.train()
print(f"Resumed model + optimizer from step {start_step}")


## 13. Inspect attention patterns

`F.scaled_dot_product_attention` does not expose its attention probabilities, so this cell recomputes
the first block's attention step by step (projections, RoPE, scores, causal mask, softmax). It also
asserts that no position attends to a future token.


In [ ]:
# ============================================================================
# 13. Inspect attention patterns
# ============================================================================
import matplotlib.pyplot as plt

prompt = "Once upon a time, a little girl named Lily"
input_ids = tokenizer.encode(prompt).ids
test_input = torch.tensor([input_ids], dtype=torch.long, device=device)

model.eval()
with torch.no_grad():
    # Recreate the first block's attention computation so we can keep the
    # attention probabilities (SDPA does not return them).
    h = model.tok_embeddings(test_input)          # (1, T, C)
    normed = model.layers[0].attention_norm(h)    # pre-attention norm

    attn = model.layers[0].attention
    B, T, C = normed.shape
    q = attn.wq(normed).view(B, T, attn.n_heads, attn.head_dim)
    k = attn.wk(normed).view(B, T, attn.n_heads, attn.head_dim)

    # Apply the same RoPE phases used during training.
    q = apply_rope(q, model.freqs_cis)
    k = apply_rope(k, model.freqs_cis)
    q = q.transpose(1, 2)  # (B, H, T, D)
    k = k.transpose(1, 2)

    # Scaled dot-product scores plus an explicit causal mask.
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(attn.head_dim)
    causal_mask = torch.triu(
        torch.full((T, T), float("-inf"), device=device), diagonal=1
    )
    weights = F.softmax(scores + causal_mask, dim=-1)  # (B, H, T, T)

    # Verify the causal property: no position attends to the future.
    assert torch.triu(weights, diagonal=1).abs().max().item() == 0.0

weights = weights[0].cpu().numpy()  # (H, T, T)
print(
    f"Attention weights: shape={weights.shape}, "
    f"mean row sum={weights.sum(axis=-1).mean():.3f}"
)

# Plot every head of the first layer in a grid.
n_cols = 4
n_rows = math.ceil(attn.n_heads / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = np.atleast_2d(axes)
for head in range(attn.n_heads):
    ax = axes[head // n_cols, head % n_cols]
    ax.imshow(weights[head], cmap="viridis", aspect="auto")
    ax.set_title(f"Layer 0 - Head {head}")
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
# Hide any unused subplots.
for ax in axes.flat[attn.n_heads :]:
    ax.axis("off")
fig.suptitle(f"PipTransformer causal attention - prompt: {prompt!r}")
fig.tight_layout()
plt.show()


## 14. Generate after training (or after loading a checkpoint)

With a trained model this produces a coherent TinyStories-style continuation. With only the sanity
check run so far, it will still be nonsense; run section 10 first for a real generation.


In [ ]:
# ============================================================================
# 14. Generate after training (or after loading a checkpoint)
# ============================================================================
prompt = "Once upon a time, a little girl named Lily"
story = generate(
    model,
    tokenizer,
    prompt,
    max_new_tokens=cfg.max_new_tokens,
    temperature=cfg.temperature,
    top_k=cfg.top_k,
    seed=SEED,
)
print("Prompt:", prompt)
print("\n--- Generated story ---")
print(story)


## 15. Optional: render the computation graph

This is illustrative only and can be slow or memory-hungry. It requires `torchviz` and the Graphviz
system package:

```bash
pip install torchviz
brew install graphviz          # macOS
# sudo apt-get install graphviz  # Debian/Ubuntu
```

Set `PLOT_ARCH = True` to render the graph to `assets/`.


In [ ]:
# ============================================================================
# 15. Optional: render the computation graph
# ============================================================================
PLOT_ARCH = False

if PLOT_ARCH:
    from torchviz import make_dot
    from IPython.display import Image, display

    os.makedirs("assets", exist_ok=True)
    x_sample = torch.zeros((1, 16), dtype=torch.long, device=device)
    out_logits = model(x_sample)
    dot = make_dot(
        out_logits,
        params=dict(model.named_parameters()),
        show_attrs=True,
        show_saved=True,
    )
    dot.format = "png"
    out_path = dot.render("assets/pip_transformer_architecture", cleanup=True)
    print(f"Saved {out_path}")
    display(Image(filename=out_path))
else:
    print("PLOT_ARCH is False; set it to True to render the graph.")


## Reference results

From the original 2,000-step run (Apple Silicon MPS, `bf16` autocast):

| Step | Validation loss | Perplexity |
| --- | --- | --- |
| 200 | 3.0136 | 20.36 |
| 600 | 2.1047 | 8.20 |
| 1000 | 1.8796 | 6.55 |
| 1400 | 1.7771 | 5.91 |
| 1800 | 1.7326 | 5.66 |
| 2000 | **1.7293** | **5.64** |

Training loss fell from `5.73` at step 50 to `1.52` at step 2000. Throughput was roughly
23k-28k tokens/s on the MPS backend.

## Notes and next steps

- **Scope:** the model is intentionally small (31.5M parameters) and trained on TinyStories only. It
  is a teaching/reference implementation, not an instruction-following assistant.
- **Tokenizer sample:** the tokenizer is trained on 50k stories while the model trains on 80k. This
  keeps the first run fast without materially hurting the vocabulary for this corpus.
- **Reproducibility:** full bitwise determinism is not guaranteed on MPS/CUDA. Seeding removes the
  most common sources of variation.
- **Scaling up:** increase `n_layers`, `dim`, `max_steps` and `train_slice` for a stronger model.
  The training loop and checkpoint format stay the same.
- **Next steps:** evaluate on held-out prompts, add a small instruction-tuning stage, or export the
  model to a standalone inference script.

## License and acknowledgements

MIT licensed (see `LICENSE`). TinyStories was introduced by Eldan and Li (2023). The architecture
draws on Llama (RoPE, SwiGLU, pre-norm) and on Andrej Karpathy's `microgpt.py`, which is included in
this repository as a separate minimal GPT implementation.
